# CT118-3-3 — Assignment 1, Task 2: Data Preparation & EDA (Final)
## Miami Housing Dataset

This is the finalised version of the Data Preparation & EDA notebook, updated with the actual
findings from the first run and the concrete pre-processing decisions those findings led to.
Every step below states **what the step does**, **why a particular decision was made based on the
real results**, the **code**, and **what to note down** for the Assignment 1 report.


## Step 0: Import Libraries

Standard imports for data handling, visualisation, and pre-processing. Unchanged from the first pass.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Step 1: Load Data & Initial Inspection

We load the CSV and inspect its shape, data types, and summary statistics before touching anything.
This confirms the dataset matches its documented size and flags any implausible values up front.

**Decision context (from the actual run):** the dataset loaded as **13,932 rows × 17 columns**, matching
the documented size exactly. `.info()` confirmed 9 `float64` and 8 `int64` columns, with no unexpected
types. `.describe()` showed `SALE_PRC` ranging from **$72,000 to $2,650,000** (plausible, but the maximum
is flagged for follow-up in Step 6 as a likely luxury-property outlier), `age` ranging from **0 to 96**
(0 = newly built, valid), `LONGITUDE` correctly negative (Western hemisphere), and `WATER_DIST` having a
minimum of **0** (interpreted as a genuine waterfront property, not an error). Overall: **no data quality
red flags at this stage** — no negative prices, no negative distances, no impossible ages.


In [ ]:
df = pd.read_csv('miami-housing.csv')  # update filename if different

print("Shape (rows, columns):", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


**Expected output / confirm:**
- Shape should read `(13932, 17)`.
- `dtypes` split should be `float64(9), int64(8)`, no `object` columns.
- `SALE_PRC` min/max should match **72000 / 2650000**; `age` min/max should match **0 / 96**.
- If your numbers differ from these, double-check you loaded the correct/unmodified CSV before continuing.


## Step 2: Handle Duplicate Records (PARCELNO)

We check how many `PARCELNO` values repeat, then check whether those repeated rows are **exact**
duplicates (safe to drop) or **genuine repeat sales** of the same property at different times/prices
(should be kept, since each row is still a valid, distinct transaction).

**Decision made:** the run found **152 repeated PARCELNO values (1.10% of unique parcels)**, matching
the ~1% noted in the dataset documentation. Critically, **0 fully duplicated rows** were found — meaning
every repeated-parcel row differs in at least one column (most likely `SALE_PRC` and/or `month_sold`).
This confirms these are **genuine repeat sales**, not data entry errors. **Decision: no rows are dropped.**
The dataset remains at 13,932 rows.


In [ ]:
dup_counts = df['PARCELNO'].value_counts()
repeated_parcels = dup_counts[dup_counts > 1]

print(f"Number of unique PARCELNO values: {df['PARCELNO'].nunique()}")
print(f"Number of PARCELNO values appearing more than once: {len(repeated_parcels)}")
print(f"Percentage of repeated parcels: {len(repeated_parcels) / df['PARCELNO'].nunique() * 100:.2f}%")

df[df['PARCELNO'].isin(repeated_parcels.index[:3])].sort_values('PARCELNO')


In [ ]:
exact_duplicates = df.duplicated().sum()
print(f"Number of fully duplicated rows (all columns identical): {exact_duplicates}")

# Decision: 0 exact duplicates found -> repeated PARCELNO values are genuine repeat sales.
# We therefore do NOT drop any rows, and simply confirm the shape is unchanged.
df = df.drop_duplicates()
print(f"Shape after (non-)dropping duplicate rows: {df.shape}")


**Expected output / confirm:**
- Unique `PARCELNO` count: **13,776**; repeated: **152 (1.10%)**.
- Exact duplicate row count: **0**.
- Shape unchanged at **(13932, 17)**.
- **For your report:** state explicitly that repeated parcels were investigated and identified as genuine
repeat sales rather than data errors, which is why no rows were removed at this step.


## Step 3: Missing Value Analysis

We check every column for missing values. This dataset turned out to be fully complete.

**Decision context:** the run found **no missing values in any column**. No imputation or row removal
was necessary. This is still reported explicitly, as a clean result is itself a finding worth stating in
the report rather than skipping.


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_summary = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_%', ascending=False)

print("Columns with missing values:")
missing_summary if not missing_summary.empty else print("No missing values found in any column.")


In [ ]:
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Heatmap')
plt.show()


**Expected output / confirm:**
- "No missing values found in any column" — the heatmap should render as a single solid colour with
no lighter "missing" streaks.
- **For your report:** state this cleanly as "no missing values were identified across all 17 features,
so no imputation strategy was required."


## Step 4: Target Variable Analysis & Transformation Decision

We examine the distribution of `SALE_PRC` itself, since a heavily skewed target can hurt neural network
training (the loss function becomes dominated by a small number of very high-value properties).

**Decision made:** the raw `SALE_PRC` skewness was **3.22** — strongly right-skewed, confirmed visually
by the histogram's long right tail and the boxplot's dense cluster of high-value outliers. After applying
a log transform (`log1p`), skewness dropped to **0.74**, much closer to a symmetric distribution.
**Decision: `log(SALE_PRC)` will be used as the modelling target for Assignment 2**, rather than raw
`SALE_PRC`. This is expected to help the network train more stably. Predictions will need to be
back-transformed with `np.expm1()` before reporting error metrics (MAE/RMSE) in real dollar terms, so
that results are interpretable to a non-technical reader.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['SALE_PRC'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of SALE_PRC')
sns.boxplot(x=df['SALE_PRC'], ax=axes[1])
axes[1].set_title('Boxplot of SALE_PRC')
plt.tight_layout()
plt.show()

print(f"Skewness of SALE_PRC: {df['SALE_PRC'].skew():.2f}")


In [ ]:
df['SALE_PRC_log'] = np.log1p(df['SALE_PRC'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['SALE_PRC_log'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of log(SALE_PRC)')
sns.boxplot(x=df['SALE_PRC_log'], ax=axes[1])
axes[1].set_title('Boxplot of log(SALE_PRC)')
plt.tight_layout()
plt.show()

print(f"Skewness of log(SALE_PRC): {df['SALE_PRC_log'].skew():.2f}")


**Expected output / confirm:**
- Raw skewness ≈ **3.22**; log-transformed skewness ≈ **0.74**.
- **For your report:** explicitly justify modelling `log(SALE_PRC)` using these two numbers, and note
that predictions must be exponentiated back (`np.expm1`) before reporting dollar-denominated metrics
in Assignment 2.


## Step 5: Univariate Analysis of Numeric Features

We plot and rank the skewness of each numeric predictor. This directly feeds into the outlier
treatment decision in Step 6 — the features with the highest skewness here are the ones most likely to
need outlier capping.

**Decision context:** `LND_SQFOOT` (**3.83**) and `SPEC_FEAT_VAL` (**1.90**) are by far the most skewed
features, well ahead of the rest (`TOT_LVG_AREA` 1.35, `WATER_DIST` 1.14, `HWY_DIST` 1.12, `RAIL_DIST`
0.82, `age` 0.51, `SUBCNTR_DI` 0.45, `CNTR_DIST` 0.43, `OCEAN_DIST` 0.39). This ranking is used directly
in Step 6/12 to decide which features receive outlier capping.


In [ ]:
numeric_cols = ['LND_SQFOOT', 'TOT_LVG_AREA', 'SPEC_FEAT_VAL', 'RAIL_DIST',
                'OCEAN_DIST', 'WATER_DIST', 'CNTR_DIST', 'SUBCNTR_DI',
                'HWY_DIST', 'age']

df[numeric_cols].hist(bins=30, figsize=(16, 12))
plt.tight_layout()
plt.show()


In [ ]:
skew_summary = df[numeric_cols].skew().sort_values(ascending=False)
print("Skewness of numeric features (sorted, highest first):")
skew_summary


**Expected output / confirm:**
- Ranking (highest first): `LND_SQFOOT` (3.83) > `SPEC_FEAT_VAL` (1.90) > `TOT_LVG_AREA` (1.35) >
`WATER_DIST` (1.14) > `HWY_DIST` (1.12) > `RAIL_DIST` (0.82) > `age` (0.51) > `SUBCNTR_DI` (0.45) >
`CNTR_DIST` (0.43) > `OCEAN_DIST` (0.39).
- **For your report:** note that `LND_SQFOOT` and `SPEC_FEAT_VAL` stand out as substantially more skewed
than the rest, which is the direct justification for treating them differently in outlier handling.


## Step 6: Outlier Detection (IQR Method)

We use the IQR method to quantify, per feature, what proportion of values fall outside
`[Q1 - 1.5×IQR, Q3 + 1.5×IQR]`. This step only **identifies** outliers — the actual capping is deferred
until **after** the train/validation/test split (Step 12), so that outlier thresholds are learned from
training data only and don't leak information from validation/test rows, exactly as we did for feature
scaling.

**Decision made, based on the actual percentages found:**
- `SPEC_FEAT_VAL` (12.22%, 1,703 rows) and `LND_SQFOOT` (9.83%, 1,370 rows) — the two features already
flagged as most skewed in Step 5 — will be **capped at the 1st/99th percentile** in Step 12. Such a high
proportion of "outliers" alongside very high skewness suggests a long tail is stretching the distribution,
rather than these being informative extreme cases.
- `SALE_PRC` (9.62%, 1,340 rows) will **not** be capped or have rows removed — these represent genuine
luxury property sales that are part of the prediction target, and the log transform from Step 4 already
substantially reduces their leverage on training.
- `TOT_LVG_AREA` (3.21%), `HWY_DIST` (2.09%), `WATER_DIST` (1.76%), `RAIL_DIST` (0.98%), and
`SUBCNTR_DI` (0.86%) are left **untouched** — under 5% outliers each, and likely reflect genuine
geographic/structural variation (e.g. some properties are simply far from a highway) rather than data
errors.
- `CNTR_DIST` (0.02%), `age` (0.02%), and `OCEAN_DIST` (0.00%) show negligible-to-no outliers and require
no action.


In [ ]:
def iqr_outlier_summary(data, columns):
    summary = []
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        n_outliers = ((data[col] < lower) | (data[col] > upper)).sum()
        summary.append({
            'feature': col, 'lower_bound': lower, 'upper_bound': upper,
            'n_outliers': n_outliers, 'pct_outliers': n_outliers / len(data) * 100
        })
    return pd.DataFrame(summary).sort_values('pct_outliers', ascending=False)

outlier_report = iqr_outlier_summary(df, numeric_cols + ['SALE_PRC'])
outlier_report


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols + ['SALE_PRC']):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()


**Expected output / confirm:**
- Outlier percentages should closely match: `SPEC_FEAT_VAL` 12.22%, `LND_SQFOOT` 9.83%, `SALE_PRC` 9.62%,
`TOT_LVG_AREA` 3.21%, `HWY_DIST` 2.09%, `WATER_DIST` 1.76%, `RAIL_DIST` 0.98%, `SUBCNTR_DI` 0.86%,
`CNTR_DIST` 0.02%, `age` 0.02%, `OCEAN_DIST` 0.00%.
- **For your report:** present this table, then state the three-tier decision above (cap two features,
leave the target alone in favour of the log transform, leave low-outlier features untouched).


## Step 7: Correlation & Multicollinearity Analysis

We examine correlations between numeric features and `SALE_PRC`, and between the features themselves,
to identify the strongest predictors and flag multicollinearity for discussion.

**Decision context:** the strongest correlates of `SALE_PRC` are `TOT_LVG_AREA` (**0.67**),
`SPEC_FEAT_VAL` (**0.50**), `SUBCNTR_DI` (**-0.37**), and `LND_SQFOOT` (**0.36**). `OCEAN_DIST` shows the
expected **negative** correlation (-0.27) — closer to the ocean, higher price — directly confirming the
hedonic pricing literature discussed in Section 5 of the report. Notable multicollinearity was found
between `CNTR_DIST` and `SUBCNTR_DI` (**0.77**), and between `LONGITUDE` and both `CNTR_DIST` (**-0.79**)
and `WATER_DIST` (**-0.76**). **Decision:** rather than dropping either of `CNTR_DIST`/`SUBCNTR_DI`
outright, we engineer a combined `AVG_CNTR_DIST` feature in Step 9 to partially address this redundancy,
while still retaining the original two distance features since neural networks are relatively robust to
multicollinearity (unlike linear regression) — this is flagged as a discussion point rather than something
that must be "fixed."


In [ ]:
corr_cols = numeric_cols + ['SALE_PRC', 'LATITUDE', 'LONGITUDE']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
price_corr = corr_matrix['SALE_PRC'].drop('SALE_PRC').sort_values(key=abs, ascending=False)
print("Feature correlation with SALE_PRC (sorted by strength):")
price_corr


**Expected output / confirm:**
- Top correlates with `SALE_PRC`: `TOT_LVG_AREA` 0.67, `SPEC_FEAT_VAL` 0.50, `SUBCNTR_DI` -0.37,
`LND_SQFOOT` 0.36, `OCEAN_DIST` -0.27, `CNTR_DIST` -0.27, `HWY_DIST` 0.23.
- Multicollinearity: `CNTR_DIST` ↔ `SUBCNTR_DI` at 0.77; `LONGITUDE` ↔ `CNTR_DIST` at -0.79.
- **For your report:** cite the `OCEAN_DIST` finding explicitly against your Section 5 literature review
(hedonic pricing / locational amenity effects), and note the multicollinearity as a discussion point.


## Step 8: Categorical, Ordinal & Binary Feature Analysis

We examine `avno60plus`, `structure_quality`, and `month_sold` with count plots and price
comparisons, since these are not continuous features.

**Decision context:** `avno60plus` is **heavily imbalanced** (mean ≈ 0.0149, i.e. only ~1.5% of
properties are flagged as exposed to excess airplane noise). Properties flagged 0 show a wider price
range with many high-value outliers, while flagged-1 properties show a narrower range with fewer
outliers — broadly consistent with the noise-discount effect from Nelson (2004), though the extreme
imbalance means this feature's practical predictive contribution may be limited, which is worth stating
plainly in the report rather than overselling. `structure_quality` shows a broadly increasing price trend
with quality level, though category 3 has a very small sample size and an unusually high median in the
boxplot — flagged as likely small-sample instability rather than a genuine effect. `month_sold` is fairly
even, with a mild peak at month 6. **Decision:** keep all three features as-is (no re-binning, no
resampling) — the imbalance and small-sample quirks are noted as limitations for discussion in
Assignment 2 rather than corrected at this stage, since artificially rebalancing raw transaction data
would misrepresent the real market.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.countplot(x='avno60plus', data=df, ax=axes[0])
axes[0].set_title('avno60plus (Airplane Noise Flag)')
sns.countplot(x='structure_quality', data=df, ax=axes[1])
axes[1].set_title('structure_quality')
sns.countplot(x='month_sold', data=df, ax=axes[2])
axes[2].set_title('month_sold')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='avno60plus', y='SALE_PRC', data=df, ax=axes[0])
axes[0].set_title('SALE_PRC by avno60plus')
sns.boxplot(x='structure_quality', y='SALE_PRC', data=df, ax=axes[1])
axes[1].set_title('SALE_PRC by structure_quality')
plt.tight_layout()
plt.show()


**Expected output / confirm:**
- `avno60plus`: roughly 13,700+ properties at 0, a small minority (~1.5%) at 1.
- `structure_quality`: category 4 most common, then 2, then 5, with 1 and 3 rare.
- `month_sold`: peak around month 6, otherwise fairly even across the year.
- **For your report:** explicitly tie the `avno60plus` price pattern back to the Nelson (2004) citation
from Section 5, and flag the small-sample category-3 anomaly in `structure_quality` as a limitation.


## Step 9: Feature Engineering & Dropping Irrelevant Columns

We drop `PARCELNO` (a pure identifier with no predictive value) and engineer `AVG_CNTR_DIST`, the mean
of `CNTR_DIST` and `SUBCNTR_DI`, as a partial response to the multicollinearity flagged in Step 7.

**Decision context:** `LATITUDE`/`LONGITUDE` are **retained** as model inputs despite correlating with
several distance features, because neural networks can tolerate multicollinearity far better than linear
models, and the raw coordinates may still let the network learn fine-grained spatial patterns the
distance features don't fully capture (e.g. neighbourhood-level effects). This is a judgement call worth
stating explicitly in the report as a considered decision, not an oversight.


In [ ]:
df_model = df.drop(columns=['PARCELNO'])

# Engineered feature: combines CNTR_DIST and SUBCNTR_DI, which showed 0.77 correlation in Step 7
df_model['AVG_CNTR_DIST'] = df_model[['CNTR_DIST', 'SUBCNTR_DI']].mean(axis=1)

print("Final columns going into modelling:")
print(df_model.columns.tolist())
df_model.head()


**Expected output / confirm:**
- Final column list should include all original features minus `PARCELNO`, plus `SALE_PRC_log` and
`AVG_CNTR_DIST`.
- **For your report:** state the rationale for keeping `LATITUDE`/`LONGITUDE` despite the correlation
findings, and describe `AVG_CNTR_DIST` as a direct response to the multicollinearity identified in
Step 7.


## Step 10: Encoding Categorical Features

We confirm `avno60plus` and `structure_quality` are already numeric and appropriately typed (binary and
ordinal respectively), so no additional encoding (e.g. one-hot encoding) is needed.

**Decision made:** confirmed both features are stored as integers with valid, expected ranges
(`avno60plus` ∈ {0, 1}; `structure_quality` ∈ {1, 2, 3, 4, 5}). **No encoding is applied** — both are used
directly as model inputs.


In [ ]:
print("avno60plus unique values:", sorted(df_model['avno60plus'].unique()))
print("structure_quality unique values:", sorted(df_model['structure_quality'].unique()))


**Expected output / confirm:**
- `avno60plus`: `[0, 1]`.
- `structure_quality`: `[1, 2, 3, 4, 5]`.
- **For your report:** a single sentence suffices: "No nominal categorical encoding was required, as
`avno60plus` is already binary and `structure_quality` is already an ordinal numeric rating."


## Step 11: Train / Validation / Test Split

We split the data **before** outlier capping and scaling, to prevent any information from the
validation/test sets leaking into decisions that should be learned from training data only.

**Decision made:** following the Step 4 decision, `y` is now set to **`SALE_PRC_log`**, not raw
`SALE_PRC`. A 70/15/15 split is used, consistent with the ratio proposed in Assignment 1 and referenced
against Retnowati et al. (2024)'s 60:20:20 finding in the literature review — the larger absolute size of
this dataset (13,932 rows vs. their smaller sample) supports allocating a slightly larger share to
training while still leaving adequate validation/test data.


In [ ]:
X = df_model.drop(columns=['SALE_PRC', 'SALE_PRC_log'])
y = df_model['SALE_PRC_log']  # Decision from Step 4: model log-price, not raw price

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, random_state=42  # 0.1765 * 0.85 ≈ 0.15
)

print(f"Train set: {X_train.shape[0]} rows ({X_train.shape[0]/len(df_model)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} rows ({X_val.shape[0]/len(df_model)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} rows ({X_test.shape[0]/len(df_model)*100:.1f}%)")


**Expected output / confirm:**
- Train: **9,751 rows (70.0%)**; Validation: **2,091 rows (15.0%)**; Test: **2,090 rows (15.0%)**.
- **For your report:** confirm these exact counts, and explicitly note that `y` is now the log-transformed
target per the Step 4 decision.


## Step 12: Outlier Treatment (Capping, Fit on Training Data Only)

Following the decision made in Step 6, we now apply 1st/99th percentile capping to `LND_SQFOOT` and
`SPEC_FEAT_VAL` — the two features flagged for their unusually high skewness (3.83 and 1.90) and high
IQR-outlier percentages (9.83% and 12.22%). The capping thresholds are learned from **`X_train` only**,
then applied to `X_train`, `X_val`, and `X_test` — mirroring the same leakage-prevention logic used for
feature scaling in Step 13, so that no information from the validation/test sets influences the
thresholds.


In [ ]:
def get_cap_bounds(series, lower_pct=0.01, upper_pct=0.99):
    return series.quantile(lower_pct), series.quantile(upper_pct)

cols_to_cap = ['LND_SQFOOT', 'SPEC_FEAT_VAL']
cap_bounds = {}

for col in cols_to_cap:
    lower, upper = get_cap_bounds(X_train[col])
    cap_bounds[col] = (lower, upper)
    print(f"{col}: capping to [{lower:.1f}, {upper:.1f}] (bounds learned from X_train only)")

for col in cols_to_cap:
    lower, upper = cap_bounds[col]
    X_train[col] = X_train[col].clip(lower, upper)
    X_val[col] = X_val[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

print("\nCapping applied to X_train, X_val, and X_test.")


**Expected output / confirm:**
- Two printed capping ranges for `LND_SQFOOT` and `SPEC_FEAT_VAL`, both derived from `X_train` only.
- **For your report:** state explicitly that capping thresholds were learned from training data only and
applied identically to validation/test data, to avoid data leakage — the same principle applied to
feature scaling.


## Step 13: Feature Scaling

We fit `StandardScaler` on the (now outlier-capped) training data only, then transform validation and
test sets using the same fitted parameters.

**Confirmed result from the actual run:** after scaling, every feature's mean was on the order of
`1e-14` to `1e-18` (effectively 0) and every feature's standard deviation was **1.000051** — confirming
correct, leakage-free scaling across all 14 input features, including the binary `avno60plus` flag, which
was scaled along with the continuous features for consistency.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

X_train_scaled.describe().T[['mean', 'std']]


**Expected output / confirm:**
- All means ≈ 0 (tiny floating-point values); all standard deviations ≈ **1.000051**.
- **For your report:** this table is strong, concrete evidence of correct scaling — include it directly.


## Step 14: Save Prepared Datasets

We save the final, capped, scaled train/validation/test sets to disk, ready for Assignment 2's model
building notebook to load directly — keeping pre-processing identical and centralised in one place.


In [ ]:
X_train_scaled.to_csv('X_train.csv', index=False)
X_val_scaled.to_csv('X_val.csv', index=False)
X_test_scaled.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_val.to_csv('y_val.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("All prepared datasets saved successfully.")
print("Note: y_train/y_val/y_test contain log-transformed SALE_PRC. Use np.expm1() to convert")
print("predictions back to dollar terms when reporting error metrics in Assignment 2.")


**Expected output / confirm:**
- Six CSV files saved: `X_train.csv`, `X_val.csv`, `X_test.csv`, `y_train.csv`, `y_val.csv`, `y_test.csv`.
- **For your report:** remind yourself (and state in Assignment 2) that the target variable saved here is
`log(SALE_PRC)`, not raw price — this single line will save you a debugging headache later.


## Step 15: Summary of Pre-Processing Decisions (for the Report)

A consolidated recap of every decision made in this notebook, useful as a checklist when writing the
Assignment 1, Task 2 write-up:

| Step | Finding | Decision |
|---|---|---|
| Duplicates | 152 repeated PARCELNO (1.10%), 0 exact duplicate rows | Keep all rows — genuine repeat sales |
| Missing values | None found in any column | No imputation needed |
| Target variable | SALE_PRC skew = 3.22 → log skew = 0.74 | Model `log(SALE_PRC)`; back-transform with `expm1` for reporting |
| Outliers: LND_SQFOOT, SPEC_FEAT_VAL | 9.83% / 12.22% IQR outliers, highest skew (3.83 / 1.90) | Cap at 1st/99th percentile, fit on training data only |
| Outliers: SALE_PRC | 9.62% IQR outliers | No capping — genuine luxury sales; handled via log transform instead |
| Outliers: other features | All under 5% | Left untouched — genuine variation |
| Multicollinearity | CNTR_DIST ↔ SUBCNTR_DI = 0.77; LONGITUDE ↔ CNTR_DIST = -0.79 | Engineered `AVG_CNTR_DIST`; retained originals (NN-robust to multicollinearity) |
| avno60plus | Heavily imbalanced (~1.5% flagged) | Kept as-is; limitation noted for discussion |
| structure_quality | Category 3 small-sample anomaly | Kept as ordinal numeric; anomaly flagged as limitation |
| Encoding | avno60plus binary, structure_quality ordinal, both already numeric | No encoding applied |
| Train/val/test split | — | 70/15/15, justified against Retnowati et al. (2024)'s 60:20:20 finding |
| Scaling | Mean ≈ 0, Std ≈ 1.000051 across all features | StandardScaler fit on training data only |

This table can be adapted directly into your Task 2 write-up as a closing summary.
